# Baseline model sesuai spesifikasi

Notebook ini membandingkan model yang tercantum di spesifikasi proyek. Cakupannya KNN, DTL dengan ID3, C4.5, dan CART, Logistic Regression, SVM, serta Naive Bayes. Setiap implementasi dari nol dipasangkan dengan pembanding scikit-learn.

Split validasi memakai seed 42 dan preprocessing dipelajari dari bagian train. Tidak ada tuning model, feature engineering, pemilihan threshold, atau penggunaan test set.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
TRAIN_PATH = ROOT / "data" / "train.csv"
if not TRAIN_PATH.exists():
    raise FileNotFoundError("Jalankan notebook dari root repository yang memiliki data/train.csv")


## Preprocessing dan DTL dari nol

Preprocessing menggunakan imputasi median dan kategori missing atau unknown yang sudah dipakai pipeline. Implementasi berikut memuat ID3, C4.5, dan CART.

In [2]:
"""Shared preprocessing and small from-scratch ID3, C4.5, and CART baselines."""

from dataclasses import dataclass
from copy import copy, deepcopy
from itertools import combinations

import numpy as np
import pandas as pd


MISSING = "__MISSING__"
UNKNOWN = "__UNKNOWN__"


class SharedPreprocessor:
    """Median-impute numeric data and retain nominal categories as strings."""

    def fit(self, X):
        self.columns = list(X.columns)
        self.numeric = [c for c in self.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.categorical = [c for c in self.columns if c not in self.numeric]
        self.numeric_fill = {
            c: X[c].median() if X[c].notna().any() else 0.0
            for c in self.numeric
        }
        self.categories = {
            c: set(X[c].dropna().astype(str).unique()) for c in self.categorical
        }
        return self

    def transform(self, X):
        result = X.reindex(columns=self.columns).copy()
        for col in self.numeric:
            result[col] = pd.to_numeric(result[col], errors="coerce").fillna(self.numeric_fill[col])
        for col in self.categorical:
            values = result[col].astype("string")
            values = values.fillna(MISSING)
            result[col] = values.where(values.isin(self.categories[col]) | values.eq(MISSING), UNKNOWN).astype(object)
        return result

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def _entropy(zeros, ones):
    total = zeros + ones
    if total == 0:
        return 0.0
    p = zeros / total
    q = ones / total
    return -(p * np.log2(p) if p else 0.0) - (q * np.log2(q) if q else 0.0)


def _gini(zeros, ones):
    total = zeros + ones
    if total == 0:
        return 0.0
    p = zeros / total
    q = ones / total
    return 1.0 - p * p - q * q


@dataclass
class Node:
    prediction: int
    positive_probability: float
    n_samples: int
    feature: str | None = None
    kind: str | None = None
    threshold: float | None = None
    categories: frozenset | None = None
    branches: dict | None = None


class BaselineTree:
    """Greedy binary classifier using information gain, gain ratio, or Gini."""

    def __init__(self, criterion="gini", max_depth=8, min_samples_leaf=40, min_samples_split=80):
        if criterion not in {"information_gain", "gain_ratio", "gini"}:
            raise ValueError("criterion must be information_gain, gain_ratio, or gini")
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self.columns = list(X.columns)
        self.numeric = [c for c in self.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.values = {c: X[c].to_numpy() for c in self.columns}
        self.y = np.asarray(y, dtype=np.int8)
        if set(np.unique(self.y)) - {0, 1}:
            raise ValueError("BaselineTree expects binary labels 0/1")
        self.root = self._grow(np.arange(len(self.y)), 0)
        return self

    def _impurity(self, zero, one):
        return _gini(zero, one) if self.criterion == "gini" else _entropy(zero, one)

    def _score(self, groups, parent_impurity, parent_size):
        if len(groups) < 2 or any(len(group) < self.min_samples_leaf for group in groups):
            return None
        weighted = sum(len(group) * self._impurity(np.sum(self.y[group] == 0), np.sum(self.y[group] == 1)) for group in groups) / parent_size
        gain = parent_impurity - weighted
        if gain <= 1e-12:
            return None
        if self.criterion != "gain_ratio":
            return gain
        split_info = _entropy(0, 0)
        for group in groups:
            p = len(group) / parent_size
            split_info -= p * np.log2(p)
        return gain / split_info if split_info else None

    def _numeric_candidates(self, indices, col, parent_impurity):
        values = self.values[col][indices].astype(float)
        order = np.argsort(values, kind="stable")
        sorted_values = values[order]
        if len(sorted_values) < 2:
            return
        cuts = np.flatnonzero(sorted_values[:-1] < sorted_values[1:]) + 1
        if len(cuts) > 64:
            cuts = np.unique(cuts[np.linspace(0, len(cuts) - 1, 64).astype(int)])
        for cut in cuts:
            if cut < self.min_samples_leaf or len(indices) - cut < self.min_samples_leaf:
                continue
            threshold = (sorted_values[cut - 1] + sorted_values[cut]) / 2
            left = indices[order[:cut]]
            right = indices[order[cut:]]
            score = self._score((left, right), parent_impurity, len(indices))
            if score is not None:
                yield score, col, "numeric", threshold, None, {"le": left, "gt": right}

    def _categorical_candidates(self, indices, col, parent_impurity):
        values = self.values[col][indices].astype(str)
        categories = np.unique(values)
        if len(categories) < 2:
            return
        if self.criterion != "gini":
            groups = {category: indices[values == category] for category in categories}
            score = self._score(tuple(groups.values()), parent_impurity, len(indices))
            if score is not None:
                yield score, col, "multiway", None, None, groups
            return

        # CART tests unique binary partitions. Above 10 levels, use one-vs-rest
        # candidates to avoid exponential work on high-cardinality features.
        if len(categories) <= 10:
            # Canonical partitions always contain the first level on the left.
            subsets = (frozenset([categories[0], *combo]) for size in range(len(categories)) for combo in combinations(categories[1:], size))
        else:
            subsets = (frozenset([category]) for category in categories)
        seen = set()
        for left_categories in subsets:
            if len(left_categories) == len(categories):
                continue
            key = tuple(sorted(left_categories))
            if key in seen:
                continue
            seen.add(key)
            left_mask = np.isin(values, list(left_categories))
            left, right = indices[left_mask], indices[~left_mask]
            score = self._score((left, right), parent_impurity, len(indices))
            if score is not None:
                yield score, col, "binary_category", None, left_categories, {"left": left, "right": right}

    def _grow(self, indices, depth):
        positives = int(self.y[indices].sum())
        negatives = len(indices) - positives
        majority = int(positives > negatives)
        node = Node(majority, positives / len(indices), len(indices))
        if (self.max_depth is not None and depth >= self.max_depth) or len(indices) < self.min_samples_split or positives == 0 or negatives == 0:
            return node
        parent_impurity = self._impurity(negatives, positives)
        best = None
        for col in self.columns:
            candidates = self._numeric_candidates(indices, col, parent_impurity) if col in self.numeric else self._categorical_candidates(indices, col, parent_impurity)
            for candidate in candidates:
                if best is None or candidate[0] > best[0]:
                    best = candidate
        if best is None:
            return node
        _, node.feature, node.kind, node.threshold, node.categories, groups = best
        node.branches = {key: self._grow(group, depth + 1) for key, group in groups.items()}
        return node

    def _predict_one(self, row):
        node = self.root
        while node.branches is not None:
            value = row[node.feature]
            if node.kind == "numeric":
                key = "le" if float(value) <= node.threshold else "gt"
            elif node.kind == "binary_category":
                key = "left" if str(value) in node.categories else "right"
            else:
                key = str(value)
            child = node.branches.get(key)
            if child is None:
                return node.prediction
            node = child
        return node.prediction

    def predict(self, X):
        return [int(self._predict_one(row)) for row in X.to_dict(orient="records")]

    def _positive_probability_one(self, row):
        node = self.root
        while node.branches is not None:
            value = row[node.feature]
            if node.kind == "numeric":
                key = "le" if float(value) <= node.threshold else "gt"
            elif node.kind == "binary_category":
                key = "left" if str(value) in node.categories else "right"
            else:
                key = str(value)
            child = node.branches.get(key)
            if child is None:
                return node.positive_probability
            node = child
        return node.positive_probability

    def predict_proba(self, X):
        positive = np.asarray([
            self._positive_probability_one(row)
            for row in X.to_dict(orient="records")
        ])
        return np.column_stack((1.0 - positive, positive))

    def _prune_node(self, node, alpha, total_samples):
        leaf_error = node.n_samples * min(node.positive_probability, 1.0 - node.positive_probability) / total_samples
        if node.branches is None:
            return leaf_error, 1

        subtree_error = 0.0
        leaves = 0
        for child in node.branches.values():
            child_error, child_leaves = self._prune_node(child, alpha, total_samples)
            subtree_error += child_error
            leaves += child_leaves

        if leaf_error + alpha <= subtree_error + alpha * leaves:
            node.feature = node.kind = node.threshold = node.categories = None
            node.branches = None
            return leaf_error, 1
        return subtree_error, leaves

    def pruned(self, alpha):
        if not np.isfinite(alpha) or alpha < 0:
            raise ValueError("alpha must be finite and non-negative")
        result = copy(self)
        result.root = deepcopy(self.root)
        self._prune_node(result.root, alpha, len(self.y))
        return result

    def leaf_count(self):
        def count(node):
            return 1 if node.branches is None else sum(count(child) for child in node.branches.values())

        return count(self.root)


## Data dan split bersama

Validasi mengambil 20 persen dari tiap kelas dengan seed 42. Statistik imputasi dan kategori hanya dipelajari dari train.

In [3]:
train_df = pd.read_csv(TRAIN_PATH, dtype={"id": str})
target = "is_canceled"
features = [column for column in train_df.columns if column not in {"id", target}]
y = train_df[target].to_numpy(dtype=np.int8)
rng = np.random.default_rng(42)
valid_idx = np.sort(np.concatenate([
    rng.permutation(np.flatnonzero(y == label))[:int(np.ceil(0.2 * np.sum(y == label)))]
    for label in (0, 1)
]))
train_idx = np.setdiff1d(np.arange(len(y)), valid_idx)
y_train, y_valid = y[train_idx], y[valid_idx]
X_train_raw, X_valid_raw = train_df.iloc[train_idx][features], train_df.iloc[valid_idx][features]
preprocessor = SharedPreprocessor().fit(X_train_raw)
X_train, X_valid = preprocessor.transform(X_train_raw), preprocessor.transform(X_valid_raw)
print(f"Train rows {len(train_idx):,}")
print(f"Validation rows {len(valid_idx):,}")
print("Validation class counts", pd.Series(y_valid).value_counts().sort_index().to_dict())


Train rows 21,898
Validation rows 5,475
Validation class counts {0: 3681, 1: 1794}


## Matriks fitur

ID3, C4.5, CART, dan Naive Bayes menerima hasil preprocessing langsung. Model lain memakai one-hot encoding untuk kategori. Fitur numerik distandardisasi untuk KNN, Logistic Regression, dan SVM.

In [4]:
if preprocessor.categorical:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    train_cat = encoder.fit_transform(X_train[preprocessor.categorical])
    valid_cat = encoder.transform(X_valid[preprocessor.categorical])
else:
    train_cat = np.empty((len(X_train), 0))
    valid_cat = np.empty((len(X_valid), 0))
numeric_train = X_train[preprocessor.numeric].to_numpy(dtype=float)
numeric_valid = X_valid[preprocessor.numeric].to_numpy(dtype=float)
train_encoded = np.column_stack([numeric_train, train_cat])
valid_encoded = np.column_stack([numeric_valid, valid_cat])
scaler = StandardScaler()
train_scaled = np.column_stack([scaler.fit_transform(numeric_train), train_cat])
valid_scaled = np.column_stack([scaler.transform(numeric_valid), valid_cat])


## Implementasi dari nol

KNN memakai jarak Euclidean dan lima tetangga. Logistic Regression memakai loss logistik dengan penalti yang dipilih. SVM linear memakai loss hinge kuadrat. SVM polynomial dan Gaussian memakai pemetaan kernel Nyström dengan 256 landmark. Pemetaannya dibuat di notebook dan menjadi aproksimasi kernel. Naive Bayes memakai distribusi Gaussian per fitur.

In [5]:
class KNNFromScratch:
    def __init__(self, k=5, batch_size=128):
        self.k = k
        self.batch_size = batch_size

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y, dtype=np.int8)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        result = []
        train_norm = np.sum(self.X * self.X, axis=1)
        for start in range(0, len(X), self.batch_size):
            batch = X[start:start + self.batch_size]
            distances = np.sum(batch * batch, axis=1)[:, None] + train_norm[None, :] - 2 * batch @ self.X.T
            nearest = np.argpartition(distances, self.k - 1, axis=1)[:, :self.k]
            votes = self.y[nearest].sum(axis=1)
            result.extend((votes * 2 >= self.k).astype(np.int8))
        return np.asarray(result)


class LogisticRegressionFromScratch:
    def __init__(self, penalty="l2", C=1.0, l1_ratio=0.5, max_iter=600, tol=1e-6):
        if penalty not in {"l1", "l2", "elasticnet"}:
            raise ValueError("Unsupported penalty")
        self.penalty = penalty
        self.C = C
        self.l1_ratio = l1_ratio
        self.max_iter = max_iter
        self.tol = tol

    @staticmethod
    def _sigmoid(values):
        return 1.0 / (1.0 + np.exp(-np.clip(values, -40, 40)))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n, p = X.shape
        alpha = 1.0 / (self.C * n)
        if self.penalty == "l1":
            l1, l2 = alpha, 0.0
        elif self.penalty == "l2":
            l1, l2 = 0.0, alpha
        else:
            l1, l2 = alpha * self.l1_ratio, alpha * (1 - self.l1_ratio)
        lipschitz = 0.25 * np.linalg.norm(X, ord=2) ** 2 / n + l2
        step = 1.0 / max(lipschitz, 1e-12)
        weights = np.zeros(p)
        bias = 0.0
        accelerated_weights = weights.copy()
        accelerated_bias = bias
        momentum = 1.0
        for _ in range(self.max_iter):
            probabilities = self._sigmoid(X @ accelerated_weights + accelerated_bias)
            gradient = X.T @ (probabilities - y) / n + l2 * accelerated_weights
            gradient_bias = np.mean(probabilities - y)
            next_weights = accelerated_weights - step * gradient
            next_weights = np.sign(next_weights) * np.maximum(np.abs(next_weights) - step * l1, 0.0)
            next_bias = accelerated_bias - step * gradient_bias
            next_momentum = (1 + np.sqrt(1 + 4 * momentum * momentum)) / 2
            next_accelerated_weights = next_weights + ((momentum - 1) / next_momentum) * (next_weights - weights)
            next_accelerated_bias = next_bias + ((momentum - 1) / next_momentum) * (next_bias - bias)
            if np.max(np.abs(next_weights - weights)) < self.tol:
                weights, bias = next_weights, next_bias
                break
            weights, bias = next_weights, next_bias
            accelerated_weights, accelerated_bias = next_accelerated_weights, next_accelerated_bias
            momentum = next_momentum
        self.weights, self.bias = weights, bias
        return self

    def predict(self, X):
        return (np.asarray(X) @ self.weights + self.bias >= 0).astype(np.int8)


def kernel_matrix(A, B, kernel, gamma, degree=3):
    dot = A @ B.T
    if kernel == "linear":
        return dot
    if kernel == "poly":
        return (gamma * dot + 1.0) ** degree
    if kernel == "rbf":
        distances = np.sum(A * A, axis=1)[:, None] + np.sum(B * B, axis=1)[None, :] - 2 * dot
        return np.exp(-gamma * np.maximum(distances, 0.0))
    raise ValueError("Unsupported kernel")


class SVMFromScratch:
    def __init__(self, kernel="linear", C=1.0, degree=3, landmarks=256, max_iter=300):
        self.kernel = kernel
        self.C = C
        self.degree = degree
        self.landmarks = landmarks
        self.max_iter = max_iter

    def _map(self, X):
        if self.kernel == "linear":
            return X
        return kernel_matrix(X, self.landmark_X, self.kernel, self.gamma, self.degree) @ self.projection

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = 2 * np.asarray(y, dtype=float) - 1
        n, p = X.shape
        self.gamma = 1.0 / max(p * np.var(X), 1e-12)
        if self.kernel != "linear":
            rng = np.random.default_rng(42)
            chosen = rng.choice(n, min(n, self.landmarks), replace=False)
            self.landmark_X = X[chosen]
            gram = kernel_matrix(self.landmark_X, self.landmark_X, self.kernel, self.gamma, self.degree)
            eigenvalues, eigenvectors = np.linalg.eigh(gram)
            keep = eigenvalues > max(eigenvalues.max() * 1e-10, 1e-12)
            self.projection = eigenvectors[:, keep] / np.sqrt(eigenvalues[keep])[None, :]
        mapped = self._map(X)
        design = np.column_stack([mapped, np.ones(n)])
        vector = np.ones(design.shape[1]) / np.sqrt(design.shape[1])
        for _ in range(20):
            vector = design.T @ (design @ vector) / n
            vector /= max(np.linalg.norm(vector), 1e-12)
        regularization = 1.0 / (self.C * n)
        lipschitz = regularization + np.dot(vector, design.T @ (design @ vector) / n)
        step = 0.9 / lipschitz
        weights = np.zeros(mapped.shape[1])
        bias = 0.0
        for _ in range(self.max_iter):
            margin = 1.0 - y * (mapped @ weights + bias)
            active = np.maximum(margin, 0.0)
            weighted = y * active
            gradient = regularization * weights - (mapped.T @ weighted) / n
            gradient_bias = -weighted.sum() / n
            next_weights = weights - step * gradient
            next_bias = bias - step * gradient_bias
            if max(np.max(np.abs(next_weights - weights)), abs(next_bias - bias)) < 1e-6:
                weights, bias = next_weights, next_bias
                break
            weights, bias = next_weights, next_bias
        self.weights, self.bias = weights, bias
        return self

    def predict(self, X):
        mapped = self._map(np.asarray(X, dtype=float))
        return (mapped @ self.weights + self.bias >= 0).astype(np.int8)


class GaussianNBFromScratch:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=np.int8)
        self.classes = np.array([0, 1], dtype=np.int8)
        self.log_prior = np.array([np.log(np.mean(y == c)) for c in self.classes])
        self.mean = np.stack([X[y == c].mean(axis=0) for c in self.classes])
        self.var = np.stack([X[y == c].var(axis=0) for c in self.classes])
        self.var += self.var_smoothing * np.var(X, axis=0).max()
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        log_probability = self.log_prior[None, :] - 0.5 * np.sum(
            np.log(2 * np.pi * self.var)[None, :, :]
            + (X[:, None, :] - self.mean[None, :, :]) ** 2 / self.var[None, :, :],
            axis=2,
        )
        return self.classes[np.argmax(log_probability, axis=1)]


## Pemeriksaan fungsi

Tes kecil ini memeriksa bentuk keluaran dan kemampuan prediksi implementasi dari nol sebelum dipakai pada data penuh.

In [6]:
toy_X = np.array([[-2.0], [-1.0], [1.0], [2.0]])
toy_y = np.array([0, 0, 1, 1], dtype=np.int8)
assert np.array_equal(KNNFromScratch(k=1).fit(toy_X, toy_y).predict(toy_X), toy_y)
for penalty in ("l1", "l2", "elasticnet"):
    assert np.array_equal(LogisticRegressionFromScratch(penalty=penalty, max_iter=3000).fit(toy_X, toy_y).predict(toy_X), toy_y)
for kernel in ("linear", "poly", "rbf"):
    assert np.array_equal(SVMFromScratch(kernel=kernel, landmarks=4, max_iter=1000).fit(toy_X, toy_y).predict(toy_X), toy_y)
assert np.array_equal(GaussianNBFromScratch().fit(toy_X, toy_y).predict(toy_X), toy_y)
print("Scratch model checks passed")


Scratch model checks passed


## Evaluasi seluruh model yang diizinkan

Metrik dihitung pada validasi yang sama. Logistic Regression memakai L1, L2, dan Elastic Net. SVM memakai kernel linear, polynomial, dan Gaussian. Pembanding CART entropy dipakai sebagai acuan ID3 dan C4.5 karena scikit-learn tidak menyediakan gain ratio.

In [7]:
rows = []

def score_model(name, family, y_pred, seconds):
    precision, recall, class_f1, _ = precision_recall_fscore_support(
        y_valid, y_pred, labels=[0, 1], zero_division=0
    )
    matrix = confusion_matrix(y_valid, y_pred, labels=[0, 1])
    rows.append({
        "model": name, "family": family,
        "accuracy": accuracy_score(y_valid, y_pred),
        "macro_f1": f1_score(y_valid, y_pred, labels=[0, 1], average="macro", zero_division=0),
        "class_0_precision": precision[0], "class_0_recall": recall[0], "class_0_f1": class_f1[0],
        "class_1_precision": precision[1], "class_1_recall": recall[1], "class_1_f1": class_f1[1],
        "tn": matrix[0, 0], "fp": matrix[0, 1], "fn": matrix[1, 0], "tp": matrix[1, 1],
        "seconds": seconds,
    })

def compare_fit(name, family, scratch_model, sklearn_model, Xs_train, Xs_valid, Xl_train, Xl_valid):
    start = time.perf_counter()
    scratch_model.fit(Xs_train, y_train)
    scratch_pred = scratch_model.predict(Xs_valid)
    score_model(name + " from scratch", family, scratch_pred, time.perf_counter() - start)
    start = time.perf_counter()
    sklearn_model.fit(Xl_train, y_train)
    sklearn_pred = sklearn_model.predict(Xl_valid)
    score_model(name + " scikit-learn", family, sklearn_pred, time.perf_counter() - start)

compare_fit("KNN k=5", "KNN", KNNFromScratch(k=5), KNeighborsClassifier(n_neighbors=5),
            train_scaled, valid_scaled, train_scaled, valid_scaled)

for penalty in ("l1", "l2", "elasticnet"):
    sklearn_penalty = LogisticRegression(
        penalty=penalty, solver="saga", C=1.0, l1_ratio=0.5 if penalty == "elasticnet" else None,
        max_iter=1500, tol=1e-4, random_state=42,
    )
    compare_fit(f"Logistic {penalty}", "Logistic Regression",
                LogisticRegressionFromScratch(penalty=penalty, C=1.0, l1_ratio=0.5), sklearn_penalty,
                train_scaled, valid_scaled, train_scaled, valid_scaled)

for kernel in ("linear", "poly", "rbf"):
    sklearn_kernel = "poly" if kernel == "poly" else kernel
    compare_fit(f"SVM {kernel}", "SVM",
                SVMFromScratch(kernel=kernel, C=1.0, degree=3, landmarks=256),
                SVC(kernel=sklearn_kernel, C=1.0, degree=3, gamma="scale", cache_size=512),
                train_scaled, valid_scaled, train_scaled, valid_scaled)

compare_fit("Gaussian Naive Bayes", "Naive Bayes",
            GaussianNBFromScratch(), GaussianNB(),
            train_encoded, valid_encoded, train_encoded, valid_encoded)

tree_specs = [
    ("ID3", "information_gain", "entropy", 0.0),
    ("C4.5", "gain_ratio", "entropy", 0.001),
    ("CART", "gini", "gini", 0.0),
]
for name, scratch_criterion, sklearn_criterion, alpha in tree_specs:
    compare_fit(
        name, "DTL",
        BaselineTree(criterion=scratch_criterion),
        DecisionTreeClassifier(criterion=sklearn_criterion, ccp_alpha=alpha, max_depth=8,
                               min_samples_split=80, min_samples_leaf=40, random_state=42),
        X_train, X_valid, train_encoded, valid_encoded,
    )

results = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results


c:\Users\aufar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,family,accuracy,macro_f1,class_0_precision,class_0_recall,class_0_f1,class_1_precision,class_1_recall,class_1_f1,tn,fp,fn,tp,seconds
0,ID3 scikit-learn,DTL,0.836895,0.812807,0.870942,0.889161,0.879957,0.762376,0.729654,0.745657,3273,408,485,1309,0.065407
1,CART from scratch,DTL,0.837443,0.812727,0.868887,0.892964,0.880761,0.767139,0.723523,0.744693,3287,394,496,1298,1.848358
2,C4.5 scikit-learn,DTL,0.835068,0.811554,0.872586,0.883727,0.878121,0.755009,0.735229,0.744987,3253,428,475,1319,0.063951
3,ID3 from scratch,DTL,0.832329,0.808621,0.871271,0.880739,0.875979,0.749715,0.732999,0.741263,3242,439,479,1315,1.155257
4,CART scikit-learn,DTL,0.833242,0.804924,0.856701,0.903015,0.879249,0.776176,0.690078,0.730599,3324,357,556,1238,0.057500
5,KNN k=5 scikit-learn,KNN,0.830137,0.803768,0.861878,0.889976,0.875702,0.758065,0.707358,0.731834,3276,405,525,1269,2.323777
6,KNN k=5 from scratch,KNN,0.830137,0.803768,0.861878,0.889976,0.875702,0.758065,0.707358,0.731834,3276,405,525,1269,1.747467
7,SVM rbf scikit-learn,SVM,0.830685,0.797298,0.842878,0.919587,0.879563,0.797121,0.648272,0.715032,3385,296,631,1163,12.753199
8,SVM poly scikit-learn,SVM,0.829406,0.792578,0.834918,0.930182,0.879979,0.812955,0.622631,0.705177,3424,257,677,1117,9.362290
9,Logistic elasticnet scikit-learn,Logistic Regression,0.800000,0.760221,0.821322,0.897854,0.857884,0.740868,0.599220,0.662558,3305,376,719,1075,12.117080


## Hasil baseline

Macro F1 menjadi skor utama sesuai kontrak proyek. Tabel juga memuat precision, recall, F1 per kelas, confusion matrix, dan waktu eksekusi.

In [8]:
results[[
    "model", "family", "accuracy", "macro_f1",
    "class_0_precision", "class_0_recall", "class_0_f1",
    "class_1_precision", "class_1_recall", "class_1_f1",
    "tn", "fp", "fn", "tp", "seconds",
]]


,model,family,accuracy,macro_f1,class_0_precision,class_0_recall,class_0_f1,class_1_precision,class_1_recall,class_1_f1,tn,fp,fn,tp,seconds
0,ID3 scikit-learn,DTL,0.836895,0.812807,0.870942,0.889161,0.879957,0.762376,0.729654,0.745657,3273,408,485,1309,0.065407
1,CART from scratch,DTL,0.837443,0.812727,0.868887,0.892964,0.880761,0.767139,0.723523,0.744693,3287,394,496,1298,1.848358
2,C4.5 scikit-learn,DTL,0.835068,0.811554,0.872586,0.883727,0.878121,0.755009,0.735229,0.744987,3253,428,475,1319,0.063951
3,ID3 from scratch,DTL,0.832329,0.808621,0.871271,0.880739,0.875979,0.749715,0.732999,0.741263,3242,439,479,1315,1.155257
4,CART scikit-learn,DTL,0.833242,0.804924,0.856701,0.903015,0.879249,0.776176,0.690078,0.730599,3324,357,556,1238,0.057500
5,KNN k=5 scikit-learn,KNN,0.830137,0.803768,0.861878,0.889976,0.875702,0.758065,0.707358,0.731834,3276,405,525,1269,2.323777
6,KNN k=5 from scratch,KNN,0.830137,0.803768,0.861878,0.889976,0.875702,0.758065,0.707358,0.731834,3276,405,525,1269,1.747467
7,SVM rbf scikit-learn,SVM,0.830685,0.797298,0.842878,0.919587,0.879563,0.797121,0.648272,0.715032,3385,296,631,1163,12.753199
8,SVM poly scikit-learn,SVM,0.829406,0.792578,0.834918,0.930182,0.879979,0.812955,0.622631,0.705177,3424,257,677,1117,9.362290
9,Logistic elasticnet scikit-learn,Logistic Regression,0.800000,0.760221,0.821322,0.897854,0.857884,0.740868,0.599220,0.662558,3305,376,719,1075,12.117080
